# BG-GCN NIDS: Baseline Evaluation Pipeline

This notebook implements a **Network Intrusion Detection System (NIDS)** using a
**BG-GCN** (GraphSAGE-based Graph Convolutional Network) baseline pipeline.

**Supported datasets:** NSL-KDD, CICIDS-2017, UAVIDS-2025

**Execution order:**
1. Run the *PyG Installation* cell (if needed).
2. Run *Imports* and *Setup* cells in order.
3. Run *Step 1* to launch the full baseline evaluation pipeline.


# PyG installation — run once if torch_geometric is not yet installed.
# Installs wheels that match the currently installed PyTorch build.
import sys
import torch

torch_base = torch.__version__.split('+')[0]
torch_build = torch.__version__.split('+')[1] if '+' in torch.__version__ else 'cpu'
pyg_wheel_url = f"https://data.pyg.org/whl/torch-{torch_base}+{torch_build}.html"
print(f"Using PyG wheel index: {pyg_wheel_url}")

!{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir torch_scatter torch_sparse torch_cluster torch_spline_conv -f {pyg_wheel_url}
!{sys.executable} -m pip install --upgrade torch-geometric

In [ ]:
# Standard library
import copy, gc, importlib, math, os, random, sys, time, tracemalloc

# Visualisation
import matplotlib.pyplot as plt

# Data science
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Scikit-learn
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import (
    LabelBinarizer,
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
)

# PyTorch Geometric (requires installation — see cell above)
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv

In [ ]:
# Sanity-check: verify PyG and supporting packages are installed.
print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')

for pkg in ['torch_geometric', 'torch_scatter', 'torch_sparse', 'torch_cluster']:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f'{pkg:<20}: {ver}')
    except Exception as exc:
        print(f"{pkg:<20}: FAILED TO LOAD ({type(exc).__name__}: {exc})")
        print('  Re-run Cell 2 to reinstall matching wheels for this torch build.')

In [ ]:
# Device configuration — options: 'auto', 'cpu', 'cuda'
DEVICE_MODE = 'auto'

if DEVICE_MODE == 'cpu':
    device = torch.device('cpu')
elif DEVICE_MODE == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError("DEVICE_MODE='cuda' but CUDA is not available.")
    device = torch.device('cuda')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Selected device mode : {DEVICE_MODE}')
print(f'Using device         : {device}')

In [ ]:
def _downcast_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to float32/int32 to halve CPU RAM usage."""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df


# NSL-KDD column names (no header in the raw files)
nsl_columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'attack', 'level',
]

nsl_train = _downcast_dataframe(pd.read_csv('./dataset/nslkdd/KDDTrain+.txt', header=None))
nsl_test = _downcast_dataframe(pd.read_csv('./dataset/nslkdd/KDDTest+.txt', header=None))
nsl_train.columns = nsl_columns
nsl_test.columns = nsl_columns
nsl_kdd_fresh = pd.concat([nsl_train, nsl_test], ignore_index=True)

cicd2017 = _downcast_dataframe(
    pd.read_csv('./dataset/cicids/Wednesday-workingHours.pcap_ISCX.csv')
)
uavids = _downcast_dataframe(pd.read_csv('./dataset/uavids/UAVIDS-2025.csv'))

# Shared registry used by all pipeline stages
datasets = {
    'CICIDS2017': cicd2017,
    'NSL-KDD': nsl_kdd_fresh,
    'UAVIDS': uavids,
}

target_columns = {
    'NSL-KDD': 'attack',
    'CICIDS2017': ' Label',
    'UAVIDS': 'label',
}

print('Loaded datasets:', {k: v.shape for k, v in datasets.items()})

## Utility Functions

In [ ]:
def set_seed(seed: int = 42) -> None:
    """Set random seed across all libraries for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def preprocess_data(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Clean inf/NaN values while keeping all columns (including target).

    Rows are dropped based on inf/NaN in numeric columns only; the original
    index is reset so callers can reindex target labels safely.
    """
    df = dataframe.copy()
    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=numeric_cols)
    return df


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Build a compact ColumnTransformer to avoid one-hot memory explosion.

    Numeric columns are standardized; categorical columns are ordinal-encoded.
    Fit on training data and apply transform() on val/test to avoid leakage.
    """
    numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

    transformers = [('num', StandardScaler(), numeric_cols)]
    if categorical_cols:
        transformers.append(
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
        )

    return ColumnTransformer(transformers=transformers, remainder='drop')

## Model Architecture

In [ ]:
class BG_GCN(torch.nn.Module):
    """BG-GCN model: Dynamic SAGEConv layers followed by a BatchNorm+Dropout MLP head.

    SAGEConv (GraphSAGE) is inductive — it generalises to unseen nodes without
    recomputing the full graph Laplacian — which is critical for classifying new,
    previously-unseen network traffic flows.

    The MLP head is appropriate for tabular node classification with no temporal dimension.
    Dropout and BatchNorm1d are applied after each SAGE layer and within the MLP
    head for regularisation and training stability.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        output_dim: int = 1,
        num_layers: int = 2,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        
        # 1. Parameterized GNN Layers using ModuleList
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        in_channels = input_dim
        for _ in range(num_layers):
            self.convs.append(SAGEConv(in_channels, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            in_channels = hidden_dim  # Output of layer n is input to layer n+1
            
        self.dropout = nn.Dropout(p=dropout)

        # 2. MLP classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def extract_features(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """Returns the latent node embeddings before the final classification head.
        Useful for downstream visualisations like t-SNE or UMAP."""
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
            x = self.dropout(x)
        return x

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # Extract graph embeddings
        x = self.extract_features(x, edge_index)
        
        # Map to class logits
        x = self.classifier(x)
        return x.squeeze(-1) if x.size(-1) == 1 else x


## Graph Construction

In [ ]:
def adaptive_graph_construction(
    X: np.ndarray,
    y: np.ndarray,
    k: int = 10,
    metric: str = 'euclidean',
    add_self_loops: bool = True,
) -> Data:
    """Build a kNN graph without an O(n^2) distance matrix.

    Uses PyNNDescent for approximate nearest-neighbour search on large datasets
    (> 1 000 samples), which scales to hundreds of thousands of samples in seconds.
    Falls back to sklearn's kneighbors_graph for small datasets or when pynndescent
    is not installed.

    Parameters
    ----------
    X : node feature matrix, shape (n, d)
    y : node labels, shape (n,)
    k : number of nearest neighbours
    metric : distance metric for kNN
    add_self_loops : if True, add a self-loop for every node

    Returns
    -------
    torch_geometric.data.Data with x, edge_index, y attributes
    """
    n = X.shape[0]
    k_eff = min(k, n - 1)
    _PYNNDESCENT_THRESHOLD = 1000  # use approximate NN for datasets above this size

    rows_list: list = []
    cols_list: list = []

    try:
        if n > _PYNNDESCENT_THRESHOLD:
            from pynndescent import NNDescent
            index = NNDescent(X, metric=metric, n_neighbors=k_eff + 1,
                              random_state=42, verbose=False)
            indices, _ = index.neighbor_graph
            # indices[:, 0] is the node itself — skip it
            indices = indices[:, 1:k_eff + 1]
            for src, neighbors in enumerate(indices):
                for dst in neighbors:
                    rows_list.append(src);      cols_list.append(int(dst))
                    rows_list.append(int(dst));  cols_list.append(src)
        else:
            raise ImportError  # fall through to sklearn for small n
    except (ImportError, Exception):
        adj = kneighbors_graph(X, n_neighbors=k_eff, metric=metric,
                               mode='connectivity', include_self=False)
        # Symmetrise to make the graph undirected
        adj = (adj + adj.T).astype(bool).astype(int)
        adj_coo = adj.tocoo()
        rows_list = adj_coo.row.tolist()
        cols_list = adj_coo.col.tolist()

    rows = np.array(rows_list, dtype=np.int64)
    cols = np.array(cols_list, dtype=np.int64)

    # Remove duplicate edges produced by symmetrisation
    edge_pairs = np.unique(np.vstack([rows, cols]), axis=1)
    rows, cols = edge_pairs[0], edge_pairs[1]

    if add_self_loops:
        self_loop_idx = np.arange(n, dtype=np.int64)
        rows = np.concatenate([rows, self_loop_idx])
        cols = np.concatenate([cols, self_loop_idx])

    edge_index = torch.tensor(np.vstack([rows, cols]), dtype=torch.long)
    return Data(
        x=torch.tensor(X, dtype=torch.float),
        edge_index=edge_index,
        y=torch.tensor(y, dtype=torch.long),
    )

## Training and Evaluation

In [ ]:
def _compute_pos_weight(y_tensor):
    """Compute pos_weight = #negatives / #positives for BCEWithLogitsLoss."""
    n_pos = float((y_tensor == 1).sum().item())
    n_neg = float((y_tensor == 0).sum().item())
    return torch.tensor([n_neg / n_pos], dtype=torch.float) if n_pos > 0 else None


def compute_class_weights(y, num_classes: int) -> torch.Tensor:
    """Compute inverse-frequency class weights from training labels.

    Returns a float32 tensor of shape [num_classes] where rarer classes
    receive higher weight, helping the model attend to minority classes.
    """
    counts = pd.Series(y).value_counts().sort_index()
    freq = counts.reindex(range(num_classes), fill_value=0).values.astype(np.float32)
    freq = np.maximum(freq, 1.0)          # avoid division by zero
    weights = freq.sum() / (num_classes * freq)  # inverse-frequency
    return torch.tensor(weights, dtype=torch.float32)


def _evaluate_split(model, graph_data, num_classes, batch_size, pos_weight=None):
    """Run batched forward pass on graph_data and return per-metric results.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model in eval mode.
    graph_data : torch_geometric.data.Data
        Graph for the split to evaluate (train / val / test).
    num_classes : int
        Number of target classes (2 = binary, >2 = multi-class).
    batch_size : int
        Mini-batch size for NeighborLoader.
    pos_weight : torch.Tensor or None
        Pre-computed pos_weight for binary BCE loss.  When provided it is
        used directly; otherwise it is recomputed from the split's labels.
        Ignored for multi-class.
    """
    loader = NeighborLoader(
        graph_data, num_neighbors=[-1], batch_size=batch_size,
        shuffle=False, num_workers=0,
    )
    all_logits, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        with torch.no_grad():
            logits = model(batch.x, batch.edge_index)
        seed_n = batch.batch_size
        all_logits.append(logits[:seed_n].detach().cpu())
        all_labels.append(batch.y[:seed_n].detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)
    labels_np = labels.numpy()

    if num_classes == 2:
        pw = pos_weight if pos_weight is not None else _compute_pos_weight(labels)
        loss_val = F.binary_cross_entropy_with_logits(
            logits.squeeze(), labels.float(), pos_weight=pw
        ).item()
        probas_pos = torch.sigmoid(logits).numpy().ravel()
        preds = (probas_pos > 0.5).astype(int)
    else:
        loss_val = F.cross_entropy(logits, labels.long()).item()
        probas = torch.softmax(logits, dim=1).numpy()
        preds = probas.argmax(axis=1)

    return {
        'Loss': loss_val,
        'Accuracy': accuracy_score(labels_np, preds),
        'Precision': precision_score(labels_np, preds, average='macro', zero_division=0),
        'Recall': recall_score(labels_np, preds, average='macro', zero_division=0),
        'F1 Score': f1_score(labels_np, preds, average='macro', zero_division=0),
    }


def train_model(
    model,
    train_graph,
    val_graph,
    num_classes,
    pos_weight,
    weights_for_loss,
    optimizer,
    scheduler,
    train_epochs,
    patience,
    min_delta,
    grad_clip,
    batch_size,
    print_every: int = 5,
):
    """Run the training loop with early stopping; return the model and elapsed time.

    Early stopping is evaluated every epoch (no_improve_count increments by 1
    per non-improving epoch).  Progress is printed every *print_every* epochs.
    """
    train_loader = NeighborLoader(
        train_graph, num_neighbors=[10, 10],
        batch_size=batch_size, shuffle=True, num_workers=0,
    )
    best_val_f1, no_improve_count, best_state = -1.0, 0, None
    train_start = time.perf_counter()

    for epoch in range(train_epochs):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out_batch = model(batch.x, batch.edge_index)
            seed_n = batch.batch_size
            out_seed, y_seed = out_batch[:seed_n], batch.y[:seed_n]

            if num_classes == 2:
                loss = F.binary_cross_entropy_with_logits(
                    out_seed.squeeze(), y_seed.float(), pos_weight=pos_weight
                )
            else:
                loss = F.cross_entropy(out_seed, y_seed.long(), weight=weights_for_loss)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        model.eval()
        val_metrics = _evaluate_split(
            model, val_graph, num_classes, batch_size, pos_weight=pos_weight
        )
        val_f1 = val_metrics['F1 Score']

        # Update best state every epoch
        if val_f1 > best_val_f1 + min_delta:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            no_improve_count = 0
        else:
            no_improve_count += 1

        if (epoch + 1) % print_every == 0 or epoch == train_epochs - 1:
            print(
                f'  Epoch {epoch + 1:>4d}/{train_epochs}: '
                f'train_loss={epoch_loss:.4f}  val_F1={val_f1:.4f}  '
                f'best_F1={best_val_f1:.4f}'
            )

        if no_improve_count >= patience:
            print(
                f'  Early stopping at epoch {epoch + 1} '
                f'(best val F1={best_val_f1:.4f})'
            )
            break

    train_time = time.perf_counter() - train_start
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_time


def evaluate_model(model, test_graph, test_y, num_classes, batch_size):
    """Run batched test inference and return a metrics dictionary."""
    test_loader = NeighborLoader(
        test_graph, num_neighbors=[-1],
        batch_size=batch_size, shuffle=False, num_workers=0,
    )
    all_test_preds, all_test_probas = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out_seed = model(batch.x, batch.edge_index)[:batch.batch_size]
            if num_classes == 2:
                probas_pos = torch.sigmoid(out_seed).cpu().numpy().ravel()
                preds = (probas_pos > 0.5).astype(int)
                all_test_probas.extend(probas_pos.tolist())
            else:
                probas = torch.softmax(out_seed, dim=1).cpu().numpy()
                preds = probas.argmax(axis=1)
                all_test_probas.extend(probas.tolist())
            all_test_preds.extend(preds.tolist())

    y_true = np.asarray(test_y)
    y_pred = np.asarray(all_test_preds).ravel()
    y_proba_for_auc = np.asarray(all_test_probas)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    confusion = confusion_matrix(y_true, y_pred)

    auc = None
    try:
        if num_classes == 2:
            auc = roc_auc_score(y_true, y_proba_for_auc)
        else:
            auc = roc_auc_score(
                y_true, y_proba_for_auc, multi_class='ovr', average='macro'
            )
    except ValueError:
        auc = None

    return {
        'accuracy': accuracy,
        'auc': auc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'report': report,
        'confusion': confusion,
    }


def run_all_benchmarks(
    train_X,
    test_X,
    train_y,
    test_y,
    train_graph,
    val_graph,
    test_graph,
    k: int = 10,
    class_weights=None,
    model_hparams=None,
):
    """Train and evaluate the BG-GCN model.

    * Gradient clipping (max_norm=1.0) every training step.
    * Validation-based early stopping with best-model restore.
    * Training uses all nodes in train_graph; evaluation uses separate val/test graphs.
    * NeighborLoader mini-batching for scalable GPU training.
    * pos_weight for binary BCE to handle class imbalance.
    * class_weights tensor for multi-class CE to handle class imbalance.
    """
    results = []
    class_names = train_y.unique() if hasattr(train_y, 'unique') else np.unique(train_y)

    lb = LabelBinarizer()
    lb.fit(train_y)

    input_dim = train_graph.x.shape[1]
    num_classes = len(class_names)
    output_dim = 1 if num_classes == 2 else num_classes

    model_hparams = model_hparams or {}
    hidden_dim = int(model_hparams.get('hidden_dim', 64))
    learning_rate = float(model_hparams.get('lr', 0.001))
    weight_decay = float(model_hparams.get('weight_decay', 0.0))
    train_epochs = int(model_hparams.get('epochs', 300))
    patience = int(model_hparams.get('patience', 20))
    min_delta = float(model_hparams.get('min_delta', 1e-4))
    grad_clip = float(model_hparams.get('grad_clip', 1.0))
    batch_size = int(model_hparams.get('batch_size', 128))
    num_layers = int(model_hparams.get('num_layers', 2))

    gnn_models = {'BG-GCN': BG_GCN(input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers)}
    confusion_matrices = {}
    split_metrics_tables = {}

    for name, model in gnn_models.items():
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        print(f'\n--- Training {name} ---')
        model = model.to(device)
        tracemalloc.start()
        start_time = time.time()

        optimizer = torch.optim.Adam(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, train_epochs)
        )

        weights_for_loss = None
        if class_weights is not None and num_classes > 2:
            weights_for_loss = class_weights.to(device)
            if weights_for_loss.numel() != output_dim:
                weights_for_loss = None

        pos_weight = None
        if num_classes == 2:
            pw = _compute_pos_weight(train_graph.y)
            pos_weight = pw.to(device) if pw is not None else None

        model, train_time = train_model(
            model=model,
            train_graph=train_graph,
            val_graph=val_graph,
            num_classes=num_classes,
            pos_weight=pos_weight,
            weights_for_loss=weights_for_loss,
            optimizer=optimizer,
            scheduler=scheduler,
            train_epochs=train_epochs,
            patience=patience,
            min_delta=min_delta,
            grad_clip=grad_clip,
            batch_size=batch_size,
        )

        model.eval()
        test_metrics = evaluate_model(
            model=model,
            test_graph=test_graph,
            test_y=test_y,
            num_classes=num_classes,
            batch_size=batch_size,
        )
        confusion_matrices[name] = test_metrics['confusion']

        split_rows = []
        for split_name, split_graph, split_train_time in [
            ('Trainset',       train_graph, train_time),
            ('Validation set', val_graph,   0.0),
            ('Test set',       test_graph,  0.0),
        ]:
            fwd_start = time.perf_counter()
            m = _evaluate_split(
                model, split_graph, num_classes, batch_size,
                pos_weight=pos_weight if num_classes == 2 else None,
            )
            fwd_time = time.perf_counter() - fwd_start
            split_rows.append({
                'Split':                split_name,
                'Train Loop Time (s)':  split_train_time,
                'Forward Time (s)':     fwd_time,
                'Eval Time (s)':        0.0,
                'Computation Time (s)': split_train_time + fwd_time,
                **m,
            })

        split_df = pd.DataFrame(split_rows)[[
            'Split', 'Train Loop Time (s)', 'Forward Time (s)', 'Eval Time (s)',
            'Computation Time (s)', 'Loss', 'Accuracy', 'Precision', 'Recall', 'F1 Score',
        ]]
        split_metrics_tables[name] = split_df

        end_time = time.time()
        mem_consumption = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()

        results.append({
            'Model':                 name,
            'Accuracy':              test_metrics['accuracy'],
            'AUC':                   test_metrics['auc'],
            'Precision':             test_metrics['precision'],
            'Recall':                test_metrics['recall'],
            'F1':                    test_metrics['f1'],
            'Classification Report': test_metrics['report'],
            'Confusion Matrix':      test_metrics['confusion'],
            'Time (s)':              f'{end_time - start_time:.2f} s',
            'Memory (MB)':           f'{mem_consumption / 1e6:.2f} MB',
        })

    gc.collect()
    torch.cuda.empty_cache()

    results_df = (
        pd.DataFrame(results)
        .drop(columns=['Classification Report', 'Confusion Matrix'])
        .sort_values('Accuracy', ascending=False)
    )
    print('\nBenchmark Results:')
    print(results_df.to_markdown(index=False))

    return results_df, confusion_matrices, split_metrics_tables


## NSL-KDD Attack Group Mapping

In [ ]:
# Fine-grained NSL-KDD attack labels grouped into five broad categories.
# Defined once here; shared by all experiment stages.
_DOS_ATTACKS = {
    'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop',
    'apache2', 'mailbomb', 'processtable', 'udpstorm',
}
_PROBING_ATTACKS = {'ipsweep', 'nmap', 'portsweep', 'satan', 'mscan', 'saint'}
_U2R_ATTACKS = {'buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack', 'xterm', 'ps'}
_R2L_ATTACKS = {
    'ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy',
    'warezclient', 'warezmaster', 'httptunnel', 'named', 'sendmail',
    'snmpgetattack', 'snmpguess', 'xlock', 'xsnoop',
}


def _map_nsl_attack_group(label: str) -> str:
    """Map a fine-grained NSL-KDD attack label to one of five broad categories.

    Returns 'normal', 'DoS', 'probing', 'U2R', 'R2L', or NaN for unknown labels.
    """
    label = str(label).strip()
    if label == 'normal':
        return 'normal'
    if label in _PROBING_ATTACKS:
        return 'probing'
    if label in _DOS_ATTACKS:
        return 'DoS'
    if label in _U2R_ATTACKS:
        return 'U2R'
    if label in _R2L_ATTACKS:
        return 'R2L'
    return np.nan

## Step 1: Baseline Evaluation Pipeline
Run the next cell to execute each dataset one-by-one:
1. Experiment with default BG-GCN parameters

Configuration is in the *Pipeline Configuration* cell below:
- `NUM_DATASETS_TO_TEST = 1`, `2`, or `'all'`
- `EXPERIMENT_CFG` for sampling
- `DEVICE_MODE = 'auto' | 'cpu' | 'cuda'` (in the Device Configuration cell)


In [ ]:
# ── Pipeline configuration ────────────────────────────────────────────────
# How many datasets to evaluate: 1, 2, or 'all'
NUM_DATASETS_TO_TEST = 'all'
DATASET_ORDER = ['CICIDS2017', 'NSL-KDD', 'UAVIDS']

BASELINE_BG_GCN_PARAMS = {
    'hidden_dim': 128,
    'lr': 0.001,
    'weight_decay': 1e-5,
    'epochs': 50,
    'k': 15,
    'threshold': 0.5,
    'num_layers': 4,
}

EXPERIMENT_CFG = {
    'new_dataset_size': 100000,
    'random_state': 42,
}


In [ ]:
def _select_dataset_names(order: list, how_many) -> list:
    """Return a subset of dataset names based on the NUM_DATASETS_TO_TEST setting."""
    if str(how_many).lower() == 'all':
        return order[:]
    n = int(how_many)
    if n not in (1, 2, len(order)):
        raise ValueError(f"NUM_DATASETS_TO_TEST must be 1, 2, or 'all'. Got: {how_many}")
    return order[:n]


def _create_imbalanced_subset(
    df: pd.DataFrame,
    target_col: str,
    new_dataset_size: int = 5000,
    random_state: int = 42,
) -> pd.DataFrame:
    """Downsample large classes while keeping small classes intact."""
    class_counts = df[target_col].value_counts()
    large_classes = class_counts[class_counts > 500]
    small_classes = class_counts[class_counts <= 500]

    total_large = max(1, large_classes.sum())
    scaled_counts = (large_classes / total_large * new_dataset_size).astype(int)

    sampled_data = []
    for class_label, original_count in large_classes.items():
        n_sample = max(1, min(int(scaled_counts[class_label]), int(original_count)))
        sampled_data.append(
            df[df[target_col] == class_label].sample(n=n_sample, random_state=random_state)
        )
    for class_label in small_classes.index:
        sampled_data.append(df[df[target_col] == class_label])

    return pd.concat(sampled_data).reset_index(drop=True)


def _prepare_dataset_for_experiment(
    dataset_name: str,
    model_hparams: dict,
    new_dataset_size: int = 5000,
    random_state: int = 42,
) -> dict:
    """Preprocess a dataset and build train/val/test graphs for a full experiment.

    * preprocess_data cleans features and target together (no alignment drift).
    * ColumnTransformer is fitted on the training split only (no data leakage).
    * Feature engineering (correlation projection + mutual-info selection) is
      computed on the training set and applied identically to val/test.
    * Separate graphs for each split — no mask bookkeeping required.
    """
    target_col = target_columns[dataset_name]
    df = preprocess_data(datasets[dataset_name].copy())

    if dataset_name == 'NSL-KDD':
        df[target_col] = df[target_col].apply(_map_nsl_attack_group)

    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    le = LabelEncoder()
    df[target_col] = le.fit_transform(df[target_col].astype(str))
    class_names = le.classes_

    df = _create_imbalanced_subset(
        df, target_col=target_col,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    X_raw = df.drop(columns=[target_col])
    y = df[target_col]

    # 60 / 20 / 20 stratified split; fall back to non-stratified if any class is too small
    try:
        train_X_raw, tmp_X_raw, train_y, tmp_y = train_test_split(
            X_raw, y, test_size=0.4, random_state=random_state, stratify=y
        )
        val_X_raw, test_X_raw, val_y, test_y = train_test_split(
            tmp_X_raw, tmp_y, test_size=0.5, random_state=random_state, stratify=tmp_y
        )
    except ValueError:
        train_X_raw, tmp_X_raw, train_y, tmp_y = train_test_split(
            X_raw, y, test_size=0.4, random_state=random_state
        )
        val_X_raw, test_X_raw, val_y, test_y = train_test_split(
            tmp_X_raw, tmp_y, test_size=0.5, random_state=random_state
        )

    # Fit preprocessing on training set only (no leakage)
    preprocessor = build_preprocessor(train_X_raw)
    train_X = pd.DataFrame(preprocessor.fit_transform(train_X_raw)).reset_index(drop=True)
    val_X = pd.DataFrame(preprocessor.transform(val_X_raw)).reset_index(drop=True)
    test_X = pd.DataFrame(preprocessor.transform(test_X_raw)).reset_index(drop=True)
    train_y = train_y.reset_index(drop=True)
    val_y = val_y.reset_index(drop=True)
    test_y = test_y.reset_index(drop=True)

    # Mutual-information feature selection (fitted on train only)
    keep_ratio = 1.0
    if keep_ratio >= 1.0:
        train_X1 = train_X
        val_X1 = val_X
        test_X1 = test_X
    else:
        mi = mutual_info_classif(train_X, train_y)
        top_features = np.argsort(mi)[-max(1, int(len(mi) * keep_ratio)):]
        train_X1 = train_X.iloc[:, top_features]
        val_X1 = val_X.iloc[:, top_features]
        test_X1 = test_X.iloc[:, top_features]

    graph_k = int(model_hparams.get('k', 10))
    # Keep full graphs on CPU to avoid CUDA OOM; batches are moved to device in train/eval loops.
    train_graph = adaptive_graph_construction(train_X1.values, train_y.values, k=graph_k)
    val_graph = adaptive_graph_construction(val_X1.values, val_y.values, k=graph_k)
    test_graph = adaptive_graph_construction(test_X1.values, test_y.values, k=graph_k)

    return {
        'train_X1': train_X1,
        'val_X1': val_X1,
        'test_X1': test_X1,
        'train_y': train_y,
        'val_y': val_y,
        'test_y': test_y,
        'train_graph': train_graph,
        'val_graph': val_graph,
        'test_graph': test_graph,
        'class_names': class_names,
    }

In [ ]:
def run_single_dataset_experiment(
    dataset_name: str,
    model_hparams: dict,
    new_dataset_size: int = 5000,
    random_state: int = 42,
) -> dict:
    """Run a full train/val/test experiment for one dataset and return results."""
    set_seed(random_state)
    bundle = _prepare_dataset_for_experiment(
        dataset_name=dataset_name,
        model_hparams=model_hparams,
        new_dataset_size=new_dataset_size,
        random_state=random_state,
    )

    num_classes = len(np.unique(bundle['train_y']))
    class_weights = None
    if num_classes > 2:
        class_weights = compute_class_weights(bundle['train_y'], num_classes)
        if dataset_name == 'CICIDS2017':
            dist = bundle['train_y'].value_counts().sort_index()
            print(f'  CICIDS2017 class distribution (train split): {dict(dist)}')
            print(
                f'  Class weights — min: {class_weights.min():.4f}, '
                f'max: {class_weights.max():.4f}'
            )

    results_df, confusion_matrices, split_metrics_tables = run_all_benchmarks(
        bundle['train_X1'], bundle['test_X1'],
        bundle['train_y'],  bundle['test_y'],
        bundle['train_graph'], bundle['val_graph'], bundle['test_graph'],
        k=10,
        class_weights=class_weights,
        model_hparams=model_hparams,
    )

    model_row = results_df.loc[results_df['Model'] == 'BG-GCN'].iloc[0]
    test_macro_f1 = float(model_row['F1'])
    test_accuracy = float(model_row['Accuracy'])

    payload = {
        'confusion_matrices': confusion_matrices,
        'class_names': bundle['class_names'],
        'split_metrics_tables': split_metrics_tables,
    }

    del bundle
    torch.cuda.empty_cache()
    gc.collect()

    return {
        'results_df':    results_df,
        'payload':       payload,
        'test_macro_f1': test_macro_f1,
        'test_accuracy': test_accuracy,
    }


In [ ]:
selected_datasets = _select_dataset_names(DATASET_ORDER, NUM_DATASETS_TO_TEST)
print(f'Selected datasets: {selected_datasets}')

dataset_results_rows = []

for idx, dataset_name in enumerate(selected_datasets, start=1):
    print(f"\n{'='*95}")
    print(f'DATASET {idx}/{len(selected_datasets)}: {dataset_name}')
    print(f"{'='*95}")

    print('\n[Step A] Baseline experiment (default params)')
    baseline_out = run_single_dataset_experiment(
        dataset_name=dataset_name,
        model_hparams=BASELINE_BG_GCN_PARAMS,
        new_dataset_size=EXPERIMENT_CFG['new_dataset_size'],
        random_state=EXPERIMENT_CFG['random_state'],
    )
    display(baseline_out['results_df'])

    print(
        f'[Result] {dataset_name} | '
        f"Test Macro-F1: {baseline_out['test_macro_f1']:.4f}, "
        f"Test Accuracy: {baseline_out['test_accuracy']:.4f}"
    )

    dataset_results_rows.append({
        'Dataset':              dataset_name,
        'Params':               str(BASELINE_BG_GCN_PARAMS),
        'Test Accuracy':        baseline_out['test_accuracy'],
        'Test Macro-F1':        baseline_out['test_macro_f1'],
    })

BASELINE_RESULTS_DF = pd.DataFrame(dataset_results_rows)
print('\nFinal summary: baseline results')
display(BASELINE_RESULTS_DF)


## Visualization

In [ ]:
# Baseline Macro-F1 per dataset
if 'BASELINE_RESULTS_DF' not in globals() or BASELINE_RESULTS_DF.empty:
    print('Run the pipeline cell first to generate BASELINE_RESULTS_DF.')
else:
    plot_df = BASELINE_RESULTS_DF.copy()
    x = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars = ax.bar(x, plot_df['Test Macro-F1'], width=0.5, label='Baseline (Default)', color='#9E9E9E')

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, min(1.03, h + 0.01),
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['Dataset'].tolist())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Test Macro-F1')
    ax.set_xlabel('Dataset')
    ax.set_title('Baseline BG-GCN: Test Macro-F1 per Dataset')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Baseline Accuracy per dataset
if 'BASELINE_RESULTS_DF' not in globals() or BASELINE_RESULTS_DF.empty:
    print('Run the pipeline cell first to generate BASELINE_RESULTS_DF.')
else:
    plot_df = BASELINE_RESULTS_DF.copy()
    x = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(max(7, len(plot_df) * 2.2), 4.2))
    bars = ax.bar(x, plot_df['Test Accuracy'], width=0.5, label='Baseline (Default)', color='#1565C0')

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, min(1.03, h + 0.01),
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['Dataset'].tolist())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Test Accuracy')
    ax.set_xlabel('Dataset')
    ax.set_title('Baseline BG-GCN: Test Accuracy per Dataset')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()
